In [1]:
import json
import os
import re
import requests
import psycopg2
import pandas as pd

MANIFEST_PATH = "../loadsmart_dbt/target/manifest.json"
CATALOG_PATH  = "../loadsmart_dbt/target/catalog.json"
MART_SCHEMA   = "analytics"

OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.1:8b"  

PG_CONN = dict(
    host="localhost",
    port=5432,
    dbname="loadsmart_dev",
    user="postgres",
    password=os.environ["DBT_PG_PASSWORD"],
)

In [2]:
def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

manifest = load_json(MANIFEST_PATH)
catalog = load_json(CATALOG_PATH)

In [3]:
def extract_models(manifest, schema=MART_SCHEMA):
    models = {}
    for unique_id, node in manifest["nodes"].items():
        if node.get("resource_type") != "model":
            continue
        if node.get("schema") != schema:
            continue
        columns = {
            col_name: {
                "description": col.get("description", "").strip(),
                "data_type": col.get("data_type"),
            }
            for col_name, col in node.get("columns", {}).items()
        }
        models[node["name"]] = {
            "unique_id": unique_id,
            "description": node.get("description", "").strip(),
            "meta": node.get("meta", {}),
            "relation": f'{node["database"]}.{node["schema"]}.{node["alias"] or node["name"]}',
            "columns": columns,
        }
    return models

models = extract_models(manifest)
print(list(models.keys()))

['dim_carrier', 'dim_date', 'dim_equipment', 'dim_lane', 'dim_shipper', 'fact_load']


In [4]:
def enrich_with_catalog(models, catalog):
    catalog_nodes = catalog["nodes"]
    for name, model in models.items():
        node = catalog_nodes.get(model["unique_id"])
        if not node:
            continue
        for col_name, col_meta in node.get("columns", {}).items():
            if col_name in model["columns"]:
                model["columns"][col_name]["data_type"] = col_meta.get("type")

enrich_with_catalog(models, catalog)

In [5]:
def extract_relationships(manifest):
    rels = []
    for unique_id, node in manifest["nodes"].items():
        if node.get("resource_type") != "test":
            continue
        meta = node.get("test_metadata", {})
        if meta.get("name") != "relationships":
            continue
        kwargs = meta.get("kwargs", {})
        args = kwargs.get("arguments", kwargs)
        depends_on = node.get("depends_on", {}).get("nodes", [])
        rels.append({
            "from_column": args.get("column_name") or kwargs.get("column_name"),
            "to_model_ref": args.get("to"),
            "to_field": args.get("field"),
            "from_model_unique_id": depends_on[0] if depends_on else None,
        })
    return rels

def clean_ref(ref_str):
    m = re.search(r"ref\('([^']+)'\)", ref_str or "")
    return m.group(1) if m else ref_str

relationships = extract_relationships(manifest)

In [6]:
import datetime as dt

def data_facts():
    conn = psycopg2.connect(**PG_CONN)
    try:
        with conn.cursor() as cur:
            cur.execute(
                f"select min(delivery_date)::date, max(delivery_date)::date "
                f"from {MART_SCHEMA}.fact_load where is_delivered"
            )
            min_d, max_d = cur.fetchone()
    finally:
        conn.close()

    is_last_day = (max_d + dt.timedelta(days=1)).month != max_d.month
    if is_last_day:
        start = max_d.replace(day=1)
    else:
        start = (max_d.replace(day=1) - dt.timedelta(days=1)).replace(day=1)
    end = (start.replace(day=28) + dt.timedelta(days=4)).replace(day=1)

    return (
        f"- Delivered loads span {min_d} to {max_d}.\n"
        f"- 'Last full month' means delivery_date >= DATE '{start}' AND delivery_date < DATE '{end}'. "
        f"Use exactly these dates; never current_date and never MAX(delivery_date) - INTERVAL.\n"
        f"- 'Across the period covered by the data' means every month from {min_d:%Y-%m} to {max_d:%Y-%m}."
    )

DATA_FACTS = data_facts()   # runs once, not on every question
print(DATA_FACTS)

- Delivered loads span 2024-01-05 to 2025-03-15.
- 'Last full month' means delivery_date >= DATE '2025-02-01' AND delivery_date < DATE '2025-03-01'. Use exactly these dates; never current_date and never MAX(delivery_date) - INTERVAL.
- 'Across the period covered by the data' means every month from 2024-01 to 2025-03.


In [7]:
def build_schema_context(models, relationships):
    lines = [
        "### SCHEMA CONTEXT",
        "Use ONLY the tables, columns, and relationships defined below.",
        ""
    ]
    
    # Pre-build lookup for relationships
    id_to_name = {v["unique_id"]: k for k, v in models.items()}
    
    for name, model in models.items():
        relation = model.get("relation", name)
        lines.append(f"### Table: `{relation}`")
        
        if model.get("description"):
            lines.append(f"Description: {model['description'].strip()}")

        # ai_hint lives inside meta in the dbt YAML, so it is pulled out of meta here
        meta = dict(model.get("meta", {}))
        hint = meta.pop("ai_hint", None)

        if hint:
            lines.append(f"AI Hint (follow these rules): {hint.strip()}")
            
        if meta:
            meta_str = ", ".join(f"{k}={v}" for k, v in meta.items())
            lines.append(f"Metadata: [{meta_str}]")
            
        lines.append("Columns:")
        for col_name, col in model.get("columns", {}).items():
            dtype = f" ({col['data_type']})" if col.get("data_type") else ""
            desc = f" — {col['description'].strip()}" if col.get("description") else ""
            lines.append(f"- `{col_name}`{dtype}{desc}")
            
        lines.append("")  # Blank line for clean section separation

    if relationships:
        lines.append("### RELATIONSHIPS")
        for r in relationships:
            from_name = id_to_name.get(r.get("from_model_unique_id"), "?")
            to_name = clean_ref(r.get("to_model_ref", ""))
            from_col = r.get("from_column", "?")
            to_field = r.get("to_field", "?")
            lines.append(f"- `{from_name}`.`{from_col}` -> `{to_name}`.`{to_field}`")

    return "\n".join(lines)

schema_context = build_schema_context(models, relationships)
print(schema_context)

### SCHEMA CONTEXT
Use ONLY the tables, columns, and relationships defined below.

### Table: `loadsmart_dev.analytics.dim_carrier`
Description: One row per carrier (trucking company).
Metadata: [layer=marts, grain=1 row per carrier_name, synonyms=carrier, transportadora, transportador]
Columns:
- `carrier_id` (text) — Surrogate key.
- `carrier_name` (text) — Carrier name; show it instead of carrier_id in answers.

### Table: `loadsmart_dev.analytics.dim_date`
Description: One row per calendar day; join on cast(fact_load.<date> as date) = date_day. Group months with year_month or date_trunc('month', date_day). Last, latest and current periods are relative to max(delivery_date) in fact_load, never current_date. Last full month: d = max(delivery_date)::date; if d is the last day of its month use that month, else the previous month; filter delivery_date >= month_start and < month_start + interval '1 month'. Period covered by the data means min to max delivery_date, every month in between.

In [8]:
def generate_sql(question, schema_context, max_d=None, model=OLLAMA_MODEL):

    prompt = f"""### Task
Generate a PostgreSQL query to answer the following question:
`{question}`

### Rules
Output
- Output ONLY one SQL statement: no comments, no explanations, no markdown fences. It must start with SELECT (or WITH when a CTE is needed) and end with a semicolon.

Tables and columns
- Use ONLY the tables, columns and relationships given in the schema below.
- ALWAYS Use the fully-qualified table names exactly as given in the schema (database.schema.table).
- Write every column as alias.column. Every table that owns a column you use MUST appear in FROM or JOIN.
- Join a table only when a column you need is not in the tables already used. Join on the keys listed in RELATIONSHIPS.
- Follow the AI Hint of each table (metric definitions, synonyms, query rules). It overrides your own assumptions.
- Give every table a short alias in FROM and JOIN, and write EVERY column as alias.column in SELECT, WHERE, GROUP BY, HAVING and ORDER BY, even when the column name exists in only one table. A column exists only in the table that lists it, and that table MUST appear in FROM or JOIN.
- Never return a surrogate key (a column ending in _id) as the answer; return the descriptive column of its dimension table.

Selecting
- SELECT only what is needed to answer the question. For a trend or a comparison, also SELECT the grouping column (period or category).

Filtering
- Every WHERE condition must come from a word, entity or period in the question. Do not add default filters.
- Words such as from, into, to, in followed by an entity or place must become WHERE conditions on the matching column.
- Add a date filter only when the question gives a period or dates. Apply exactly the dates written in the question or in Data facts; never compute dates yourself.

Aggregation
- A metric column in an aggregated query MUST be wrapped in an aggregate function (SUM, AVG, MIN, MAX, COUNT).
- Every non-aggregated column in SELECT, HAVING or ORDER BY MUST be in GROUP BY.
- Never divide one column by another unless the question asks for a ratio or "per <unit>". Then use SUM(a) / NULLIF(SUM(b), 0).
- Never put an unaggregated expression next to a GROUP BY.

Ranking
- For "which X has the most / least / highest / lowest / top N": GROUP BY only X, ORDER BY the aggregate (DESC for most, highest, top; ASC for least, lowest), and LIMIT 1 (LIMIT N for "top N").

Comparisons
- For "compare", "vs", "versus", "differ", "difference between": return ONE ROW PER compared value. GROUP BY the column that holds the compared values and SELECT it next to the aggregate.
- If the question names the values, use WHERE column IN ('A','B') together with GROUP BY column. If the values are the categories of a column, GROUP BY that column with no WHERE on it.
- Never repeat the same aggregate under different aliases. Never filter to only one of the compared values.

Trends over time
- For a trend or a change over time, return one row per period (period, metric) and add LAG(metric) OVER (ORDER BY period) for the change. Never one column per period.
- Never hard-code a period that is not in the question or in Data facts.

### Data facts
{DATA_FACTS}

### Database Schema
The query will run on a database with the following schema. Only use these tables and columns.
{schema_context}

### SQL Query
```sql
"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=600,
    )
    response.raise_for_status()
    raw = response.json()["response"]

    # (changed) also accepts WITH, so CTE queries are not cut off
    match = re.search(r"\b(?:with|select)\b.*?;", raw, re.IGNORECASE | re.DOTALL)
    if match:
        sql = match.group(0)
    else:
        match = re.search(r"\b(?:with|select)\b.*", raw, re.IGNORECASE | re.DOTALL)
        sql = (match.group(0).strip() + ";") if match else raw.strip()

    return sql.strip()

In [9]:
teste_sql = generate_sql("Quantos loads foram entregues no total?", schema_context)
print(teste_sql)

SELECT COUNT(*) FROM fact_load WHERE is_delivered = TRUE;


In [10]:
def execute_sql(sql, limit=200):
    conn = psycopg2.connect(**PG_CONN)
    try:
        with conn.cursor() as cur:
            cur.execute(f"SET search_path TO {MART_SCHEMA}, public;")
        df = pd.read_sql(sql, conn)
        return df.head(limit)
    finally:
        conn.close()

execute_sql(teste_sql)

C:\Users\mariana.kober_involv\AppData\Local\Temp\ipykernel_20568\3415593529.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


,count
0,4811


In [11]:
def answer_question(question):
    try:
        sql = generate_sql(question, schema_context)
    except requests.exceptions.ReadTimeout:
        return {"question": question, "sql": None, "result": None, "error": "Ollama timeout ao gerar SQL"}
    try:
        df = execute_sql(sql)
        return {"question": question, "sql": sql, "result": df, "error": None}
    except Exception as e:
        return {"question": question, "sql": sql, "result": None, "error": str(e)}
questions = [
    "For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?"
]

results = [answer_question(q) for q in questions]
for r in results:
    print("—" * 60)
    print("Q:", r["question"])
    print("SQL:", r["sql"])
    print("Erro:", r["error"])

————————————————————————————————————————————————————————————
Q: For the shipper with the most delivered loads, how did monthly volume change across the period covered by the data?
SQL: WITH top_shipper AS (
  SELECT shipper_id
  FROM loadsmart_dev.analytics.fact_load
  WHERE is_delivered = TRUE
  GROUP BY shipper_id
  ORDER BY COUNT(*) DESC
  LIMIT 1
),
monthly_volume AS (
  SELECT 
    to_char(f.delivery_date, 'YYYY-MM') AS month,
    COUNT(*) AS volume
  FROM loadsmart_dev.analytics.fact_load f
  JOIN top_shipper ts ON f.shipper_id = ts.shipper_id
  WHERE f.is_delivered = TRUE
  GROUP BY month
),
monthly_change AS (
  SELECT 
    month,
    volume,
    LAG(volume) OVER (ORDER BY month) AS prev_volume,
    volume - LAG(volume) OVER (ORDER BY month) AS change
  FROM monthly_volume
)
SELECT 
  month,
  volume,
  change
FROM monthly_change
ORDER BY month;
Erro: None


C:\Users\mariana.kober_involv\AppData\Local\Temp\ipykernel_20568\3415593529.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn)


In [12]:
results_table = pd.DataFrame([
    {
        "question": r["question"],
        "generated_sql": r["sql"],
        "answer": r["result"].to_dict(orient="records") if r["result"] is not None else None,
        "error": r["error"],
    }
    for r in results
])
results_table.to_csv("results_table.csv", index=False)
results_table

,question,generated_sql,answer,error
0,"For the shipper with the most delivered loads,...",WITH top_shipper AS (\n SELECT shipper_id\n ...,"[{'month': '2024-01', 'volume': 126, 'change':...",None
